# 01 · EDA del bronze de OEFA: la criba de preguntas

**Laboratorio en Python, no sistema.** La carga, la limpieza y el K-Means del proyecto son Go
(`cmd/cargar`, `cmd/entrenar`), y Go es la fuente de verdad. Este notebook explora el CSV largo de
OEFA para decidir **qué pregunta puede responder el dato**, con un prototipo de la limpieza que
replica la de Go (misma config, misma cascada de reglas, mismos nombres) y que después sirve de
auditoría.

Qué hace, en orden:

1. Mira el **bronze**: primer megabyte (BOM, delimitador, comillas), tamaño y sha256.
2. Lo lee con `configs/limpieza.json` (el mismo JSON que lee `cmd/cargar`, esquema de
   `datos.ConfigLimpieza`), aplica la **cascada de reglas de Go** con sus conteos y escribe un
   **silver prototipo** con el contrato de Go (byte a byte igual al de `cmd/cargar`).
3. Muestra los valores reales que hay que confirmar: parámetros sin canónico, periodos, unidades.
4. Hace **cada medición de la criba** (preguntas candidatas, §2): rango de fechas y años completos;
   completitud estación × parámetro; huecos y el PM de las 6 h previas a cada hueco; horas sobre el
   ECA y su promedio móvil de 24 h; PM2.5 ≤ PM10 y fracción fina; distancias entre estaciones;
   ciclos diario y semanal; meteorología junto con PM; coordenadas y departamento.
5. Escribe `reports/eda_criba.md` con una sección por candidata **Q1–Q9**: *pasa / no pasa / dudoso*
   y la cifra que lo sostiene.

| Entra | Sale |
|---|---|
| `data/bronze/*.csv` (o `AIRE_BRONZE`) | `data/silver/oefa_ancho_prototipo.csv` |
| `configs/limpieza.json`, `configs/kmeans.json` | `reports/eda_criba.md`, `eda_bronze.json`, `eda_limpieza_prototipo.json` (esquema de `reports/limpieza.json`) |
| | `reports/eda_completitud.csv`, `eda_estaciones.csv`, `reports/figuras_eda/*.png` |
| | figuras del informe: `completitud.png` y `distribuciones.png` |

Specs: `docs/specs/2026-10-31-preguntas-candidatas.md` (§1 y §2) y `docs/specs/2026-10-31-pc3-design.md`
(§4). Los umbrales de la criba están juntos en una celda (§12) para ajustarlos viendo los datos.

## 0. Entorno

In [ ]:
# Colab trae casi todo; esta celda instala solo lo que falte (fuera de Colab no hace nada si ya está).
import importlib.util, subprocess, sys
PAQUETES = {"pandas": "pandas", "numpy": "numpy", "matplotlib": "matplotlib"}
faltan = [pip for pip, mod in PAQUETES.items() if importlib.util.find_spec(mod) is None]
if faltan:
    print("Instalando:", faltan)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)

In [ ]:
# ── Entorno: Drive, raíz de las capas y código compartido (notebooks/aire_lab.py) ──────────────
import os, sys, subprocess
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:  # fuera de Colab no se monta nada y la celda no falla
    from google.colab import drive
    drive.mount("/content/drive")

# Parámetros de ubicación: lo único que hay que tocar.
RAIZ_DRIVE = Path("/content/drive/MyDrive/aire-distribuido")  # capas medallón en Drive
RAIZ_LOCAL = os.environ.get("AIRE_RAIZ")   # alternativa local (p. ej. el sintético); None = repo
REPO_GIT = "https://github.com/R0SEWT/aire-distribuido.git"
RAMAS = ["feature/spec-pc3", "develop"]   # de dónde clonar aire_lab.py en Colab, en ese orden
ACTUALIZAR_CODIGO = False                  # True: vuelve a clonar (si cambió aire_lab.py)


def ubicar_repo() -> Path:
    for c in (Path.cwd(), Path.cwd().parent):
        if (c / "notebooks" / "aire_lab.py").exists():
            return c
    destino = Path("/content/aire-distribuido")
    if ACTUALIZAR_CODIGO:
        subprocess.run(["rm", "-rf", str(destino)], check=False)
    for rama in RAMAS:
        if (destino / "notebooks" / "aire_lab.py").exists():
            break
        subprocess.run(["rm", "-rf", str(destino)], check=False)
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", rama, REPO_GIT, str(destino)],
                       check=False)
    if not (destino / "notebooks" / "aire_lab.py").exists():
        raise FileNotFoundError("No encuentro notebooks/aire_lab.py: clona el repo o súbelo junto al notebook.")
    return destino


REPO = ubicar_repo()
sys.path.insert(0, str(REPO / "notebooks"))
RAIZ = Path(RAIZ_LOCAL) if RAIZ_LOCAL else (RAIZ_DRIVE if EN_COLAB else REPO)
DATA, REPORTS = RAIZ / "data", RAIZ / "reports"
CONFIGS = [RAIZ / "configs", REPO / "configs"]  # la de la raíz (Drive), si existe, va primero
# Figuras del informe: en local, docs/informe/generado/figuras/; en Colab, reports/figuras/ de Drive.
FIGURAS = Path(os.environ["AIRE_FIGURAS"]) if os.environ.get("AIRE_FIGURAS") else (
    RAIZ / "reports" / "figuras" if EN_COLAB else REPO / "docs" / "informe" / "generado" / "figuras")
REPORTS.mkdir(parents=True, exist_ok=True)
FIGURAS.mkdir(parents=True, exist_ok=True)

import importlib
import aire_lab as al
importlib.reload(al)
CMAPS = al.estilo_figuras()
SINTETICO = al.es_sintetico(RAIZ)  # se vuelve a mirar al leer los datos (estaciones SINT-/SIN-)
print(f"Colab: {EN_COLAB}\nrepo: {REPO}\nraíz de las capas: {RAIZ}\nfiguras del informe: {FIGURAS}")

In [ ]:
import json, math, time, datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option("display.max_rows", 80)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

FIG_EDA = REPORTS / "figuras_eda"
FIG_EDA.mkdir(parents=True, exist_ok=True)


def guardar(fig, nombre, informe=False):
    """Guarda en reports/figuras_eda/ y, si es del informe, también en FIGURAS (con su origen)."""
    al.marca_sintetico(fig, SINTETICO)
    fig.savefig(FIG_EDA / nombre, dpi=150)
    if informe:
        fig.savefig(FIGURAS / nombre, dpi=150)
        al.registrar_figura(FIGURAS, nombre, SINTETICO, f"bronze {BRONZE.name}", "01_eda_criba")
    plt.show()

## 1. El bronze: primer megabyte, tamaño y sha256

Lo primero que se mira del CSV (spec PC3, §4.1): cabecera, BOM, delimitador, codificación y si hay
campos entrecomillados con saltos de línea dentro, que romperían la partición por bloques del
cargador de Go.

In [ ]:
BRONZE = os.environ.get("AIRE_BRONZE")  # None: el CSV más grande de data/bronze/
CALCULAR_SHA256 = True                  # 1.3 GB: unos segundos

dir_bronze = DATA / "bronze"
if BRONZE is None:
    candidatos = sorted(dir_bronze.glob("*.csv"), key=lambda p: p.stat().st_size, reverse=True)
    if not candidatos:
        raise FileNotFoundError(f"No hay ningún CSV en {dir_bronze}: sube el bronze de OEFA a Drive.")
    BRONZE = candidatos[0]
BRONZE = Path(BRONZE)
SINTETICO = SINTETICO or al.es_sintetico(bronze=BRONZE)

primer_mb = al.inspeccionar_primer_mb(BRONZE)
info_bronze = {
    "archivo": BRONZE.name,
    "bytes": BRONZE.stat().st_size,
    "modificado": dt.datetime.fromtimestamp(BRONZE.stat().st_mtime).isoformat(timespec="seconds"),
    "sha256": al.sha256(BRONZE) if CALCULAR_SHA256 else None,
    "primer_mb": {k: v for k, v in primer_mb.items() if k != "primeras_lineas"},
}
(REPORTS / "eda_bronze.json").write_text(json.dumps(info_bronze, indent=2, ensure_ascii=False, default=str),
                                         encoding="utf-8")
print(f"{BRONZE}: {info_bronze['bytes']:,} bytes · sha256 {info_bronze['sha256']}")
for k in ["bom_utf8", "codificacion_probable", "delimitador_probable", "fin_de_linea", "hay_comillas",
          "lineas_con_comillas_impares", "columnas_por_fila"]:
    print(f"  {k}: {primer_mb[k]}")
print("cabecera:", primer_mb["cabecera"])
for linea in primer_mb["primeras_lineas"][:3]:
    print("  ", linea)
if primer_mb["lineas_con_comillas_impares"]:
    print("AVISO: hay líneas con comillas impares: algún campo trae saltos de línea. "
          "El lector de Go no puede partir por saltos de línea.")
if SINTETICO:
    print("AVISO: datos sintéticos (muestra de data/sample, sintetico.py o estaciones SINT-/SIN-): "
          "las cifras prueban el notebook, no dicen nada del aire del Perú.")

## 2. Config de limpieza y lectura

`configs/limpieza.json` es el mismo archivo que lee `cmd/cargar`, con su esquema
(`datos.ConfigLimpieza`, documentado en `datos/doc.go`): como en Go, el JSON es la config entera
(lo que falta vale vacío) y un campo desconocido es un error. Si no existe, se usa
`aire_lab.CONFIG_DEFECTO` (= `datos.ConfigPorDefecto()`). `AJUSTES` pisa a la config para probar un
cambio aquí antes de llevarlo al JSON. Las columnas se buscan por nombre normalizado, como en Go; si
falta una, el error sugiere un nombre parecido de la cabecera.

In [ ]:
AJUSTES = {}   # p. ej. {"columnas": {"estacion": "ESTACION"}, "formatos_fecha": ["2006-01-02 15:04:05"]}

cfg, origen_cfg = al.cargar_config([c / "limpieza.json" for c in CONFIGS])
cfg = al.fusionar(cfg, AJUSTES)
print("Config de limpieza:", origen_cfg, "+ AJUSTES" if AJUSTES else "")
for s in cfg.get("_supuestos", []):
    print("  supuesto:", s)
if cfg.get("delimitador") not in ("", "auto", None) and primer_mb["delimitador_probable"] != cfg["delimitador"]:
    print(f"AVISO: la cabecera sugiere el delimitador {primer_mb['delimitador_probable']!r} "
          f"y la config dice {cfg['delimitador']!r}.")

t0 = time.perf_counter()
largo, info_lectura = al.leer_bronze(BRONZE, cfg)
print(f"{info_lectura['leidas']:,} filas largas leídas en {time.perf_counter() - t0:.1f} s "
      f"({largo.memory_usage(deep=True).sum() / 2**20:,.0f} MiB en memoria) · "
      f"BOM {info_lectura['bom']} · delimitador {info_lectura['delimitador']!r}")
display(pd.Series(info_lectura["columnas"], name="columna real").to_frame())
for a in info_lectura["avisos"]:
    print("AVISO:", a)
sin_usar = [c for c in info_lectura["cabecera"] if c not in info_lectura["columnas"].values()]
print("Columnas del CSV que no se usan:", sin_usar)

## 3. Limpieza prototipo: la cascada de Go y el silver prototipo

Cada fila descartada cuenta en la **primera** regla que falla, en el orden de `datos/bloque.go`:
columnas → estación vacía → periodo → parámetro no mapeado → fecha → fecha fuera de rango → valor no
numérico → centinela (sobre el valor crudo) → unidad → negativo → (factor de la unidad y del mapeo)
→ fuera de rango (ya en la unidad canónica) → duplicados (se conserva el primero del archivo;
los siguientes son `duplicado_exacto` o `duplicado_valor_distinto`, que además marca la celda).
Se cumple leídas = procesadas + Σ descartadas, como en `reports/limpieza.json`.

In [ ]:
t0 = time.perf_counter()
validos, rep_limpieza = al.limpiar(largo, cfg, info_lectura)
print(f"Cascada en {time.perf_counter() - t0:.1f} s")
fl = rep_limpieza["filas_largas"]
tabla_reglas = pd.DataFrame([(r, d, fl["descartadas_por_regla"][r]) for r, d in al.REGLAS],
                            columns=["regla", "qué", "filas descartadas"])
tabla_reglas["% de las leídas"] = 100 * tabla_reglas["filas descartadas"] / max(fl["leidas"], 1)
display(tabla_reglas)
print(f"Leídas {fl['leidas']:,} = procesadas {fl['procesadas']:,} + descartadas {fl['descartadas']:,}")
print("Marcadas (se conservan):", fl["marcadas_por_regla"])
print("Fuera de rango por parámetro:", fl["fuera_de_rango_por_parametro"])
for r, ej in rep_limpieza["_python"]["ejemplos_por_regla"].items():
    if ej:
        print(f"  ejemplos de {r}: {ej}")

**Valores reales que hay que confirmar el lunes** (lo mismo que `vistos` en `reports/limpieza.json`):
parámetros y su canónico (los que no tienen se descartan: si alguno importa, se agrega a
`parametros`), valores de periodo (cuáles cuentan como horarios) y unidades por parámetro (una
unidad que no figura en `unidades` descarta la fila).

In [ ]:
reg_cfg = al.compilar(cfg)
vistos = rep_limpieza["vistos"]
tv = pd.Series(vistos["parametros"], name="filas").sort_values(ascending=False).to_frame()
tv["canonico"] = [(reg_cfg["parametros"].get(al.normalizar(x)) or (None,))[0] for x in tv.index]
display(tv)
if vistos["periodos"]:
    tp = pd.Series(vistos["periodos"], name="filas").sort_values(ascending=False).to_frame()
    tp["cuenta_como_horario"] = [al.normalizar(x) in reg_cfg["periodos"] for x in tp.index]
    display(tp)
filas_u = [(p, u, c, (p not in reg_cfg["unidades"]) or (al.normalizar(u) in reg_cfg["unidades"][p]))
           for p, us in vistos["unidades"].items() for u, c in us.items()]
display(pd.DataFrame(filas_u, columns=["parametro", "unidad", "filas", "aceptada"]).sort_values(["parametro", "filas"], ascending=[True, False]))

In [ ]:
ancho = al.pivotar(validos, rep_limpieza)
PARAMS = [c for c in ancho.columns if c not in ("estacion", "fecha_hora")]
ESTACIONES = sorted(ancho["estacion"].unique())
N_EST = len(ESTACIONES)
SINTETICO = SINTETICO or al.es_sintetico(estaciones=ESTACIONES)
print(f"Silver prototipo: {len(ancho):,} filas anchas · {N_EST} estaciones · parámetros {PARAMS}")
print(f"Celdas con dato: {int(ancho[PARAMS].notna().sum().sum()):,} (= procesadas {fl['procesadas']:,}) · "
      f"PM2.5 > PM10 en {rep_limpieza['filas_anchas']['pm2_5_mayor_pm10']} filas")
assert int(ancho[PARAMS].notna().sum().sum()) == fl["procesadas"]

ruta_silver = DATA / "silver" / "oefa_ancho_prototipo.csv"
al.escribir_csv_contrato(ancho, ruta_silver)
print("Escrito", ruta_silver)
if (DATA / "silver" / "oefa_ancho.csv").exists():
    igual = al.sha256(ruta_silver) == al.sha256(DATA / "silver" / "oefa_ancho.csv")
    print("¿Idéntico byte a byte al silver de Go?", "sí" if igual else "NO: ver la auditoría del notebook 03")

# Regla 8 para la receta de configs/kmeans.json y las variantes de la calibración: filas completas por estación.
try:
    receta, origen_receta = al.leer_receta([c / "kmeans.json" for c in CONFIGS])
except FileNotFoundError:
    receta, origen_receta = {"features": ["pm10", "fraccion_fina", "viento_velocidad", "humedad"],
                             "log1p": ["pm10"], "nivel": "absoluto", "k": 5}, "receta provisional"
CONJUNTOS = {
    "F1-M0 (pm2_5, pm10)": (["pm2_5", "pm10"], ["pm2_5", "pm10"]),
    "F2-M0 (pm10, fracción)": (["pm10", "fraccion_fina"], ["pm10"]),
    "F2-M1 (+ viento, humedad)": (["pm10", "fraccion_fina", "viento_velocidad", "humedad"], ["pm10"]),
    "F2-M2 (+ temperatura, presión)": (["pm10", "fraccion_fina", "viento_velocidad", "humedad", "temperatura", "presion"], ["pm10"]),
}
regla8 = pd.DataFrame({n: ancho.loc[al.completas(ancho, f, l), "estacion"].value_counts().reindex(ESTACIONES, fill_value=0)
                       for n, (f, l) in CONJUNTOS.items()})
regla8.loc["TOTAL"] = regla8.sum()
display(regla8)

gold_proto, _ = al.construir_gold(ancho, receta)
al.completar_reporte(rep_limpieza, gold_proto, receta)
print(f"Receta ({origen_receta}): {receta['features']} → {len(gold_proto):,} filas completas para el gold")
(REPORTS / "eda_limpieza_prototipo.json").write_text(
    json.dumps(rep_limpieza, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

## 4. Estaciones: coordenadas, departamento y distancias

Una estación puede aparecer con más de un juego de coordenadas (se movió el equipo, o un error de
digitación): se toma el más frecuente y se cuenta cuántos hay. UTM (Este, Norte, Zona) → lat/lon en
WGS84; un datum PSAD56 desplaza unos 300–400 m, irrelevante para agrupar a 15 km.

In [ ]:
# Ubicación de cada estación: la de su primera fila, como Go (reports/limpieza.json, estaciones.*.ubicacion).
ok_filas = largo["_campos"].to_numpy() == info_lectura["total_columnas"]
cols_ubic = [c for c in al.COLUMNAS_UBICACION if c in largo]
estaciones = pd.DataFrame({e: r.get("ubicacion", {}) for e, r in rep_limpieza["estaciones"].items()}).T
estaciones.index.name = "estacion"
if cols_ubic:
    combos = largo.loc[ok_filas, ["estacion"] + cols_ubic].astype(str).apply(lambda s: s.str.strip())
    estaciones["juegos_de_ubicacion"] = combos.drop_duplicates().groupby("estacion").size().reindex(estaciones.index)
else:
    estaciones["juegos_de_ubicacion"] = np.nan
if {"este", "norte", "zona"} <= set(estaciones.columns):
    este = np.array([al.parsear_valor_go(str(x)) for x in estaciones["este"]])
    norte = np.array([al.parsear_valor_go(str(x)) for x in estaciones["norte"]])
    zonas = [al.zona_utm(z) for z in estaciones["zona"]]
    lat, lon = al.utm_a_latlon(este, norte, [z[0] for z in zonas], [z[1] for z in zonas])
    estaciones["lat"], estaciones["lon"] = np.round(lat, 5), np.round(lon, 5)
else:
    estaciones["lat"] = estaciones["lon"] = np.nan
    print("AVISO: faltan columnas de coordenadas; no hay distancias.")
estaciones["con_datos_validos"] = estaciones.index.isin(ESTACIONES)
estaciones["region"] = [al.region(d) for d in estaciones.get("departamento", pd.Series(index=estaciones.index, dtype=object))]
estaciones.to_csv(REPORTS / "eda_estaciones.csv")
display(estaciones)
print("Estaciones con más de un juego de ubicación:", list(estaciones.index[estaciones["juegos_de_ubicacion"] > 1]))

In [ ]:
DIST_GRUPO_KM = 15
con_coord = estaciones.dropna(subset=["lat", "lon"])
nombres = list(con_coord.index)
la, lo = con_coord["lat"].to_numpy(), con_coord["lon"].to_numpy()
dist = pd.DataFrame(al.haversine_km(la[:, None], lo[:, None], la[None, :], lo[None, :]), index=nombres, columns=nombres)

# Días con PM en cada estación, para medir el solape de las parejas cercanas.
con_pm = ancho[[c for c in ["pm2_5", "pm10"] if c in ancho]].notna().any(axis=1)
dias_pm = ancho.loc[con_pm].assign(dia=lambda d: d["fecha_hora"].dt.normalize()).groupby("estacion")["dia"].agg(set)
pm_dia = (ancho.assign(dia=ancho["fecha_hora"].dt.normalize())
          .groupby(["estacion", "dia"])[[c for c in ["pm10", "pm2_5"] if c in ancho]].mean())
parejas = []
for i, a in enumerate(nombres):
    for b in nombres[i + 1:]:
        d = dist.loc[a, b]
        if d < DIST_GRUPO_KM:
            solape = len(dias_pm.get(a, set()) & dias_pm.get(b, set()))
            r = np.nan
            col = "pm10" if "pm10" in pm_dia else "pm2_5"
            if a in pm_dia.index.get_level_values(0) and b in pm_dia.index.get_level_values(0):
                j = pd.concat([pm_dia.loc[a, col], pm_dia.loc[b, col]], axis=1, keys=["a", "b"], sort=True).dropna()
                r = j["a"].corr(j["b"]) if len(j) > 10 else np.nan
            parejas.append({"a": a, "b": b, "km": round(float(d), 2), "dias_con_pm_en_ambas": solape,
                            f"correlacion_diaria_{col}": None if np.isnan(r) else round(float(r), 3)})
parejas = pd.DataFrame(parejas)
grupos15 = al.grupos_cercanos(dist, 15)
grupos10 = al.grupos_cercanos(dist, 10)
print(f"Parejas a menos de {DIST_GRUPO_KM} km: {len(parejas)}")
display(parejas)
print("Grupos a < 15 km:", grupos15)
print("Grupos a < 10 km:", grupos10)
if len(nombres) > 1:
    vecino = dist.where(~np.eye(len(nombres), dtype=bool)).min(axis=1)
    print(f"Distancia a la estación más cercana: mediana {vecino.median():.1f} km, máximo {vecino.max():.1f} km")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 7.5))
ax.scatter(con_coord["lon"], con_coord["lat"], s=42, color=al.PALETA[0], edgecolor=al.SUPERFICIE, linewidth=1.5, zorder=3)
for g in grupos15:
    sub = con_coord.loc[g]
    ax.plot(sub["lon"], sub["lat"], color=al.TINTA_2, linewidth=0.8, zorder=2)
for n, f in con_coord.iterrows():
    ax.annotate(f"{n} ({al.NOMBRE_REGION.get(f['region'], f['region'])})", (f["lon"], f["lat"]), fontsize=6.5,
                color=al.TINTA_2, xytext=(4, 3), textcoords="offset points")
ax.set_xlabel("Longitud (°)")
ax.set_ylabel("Latitud (°)")
ax.set_title("Estaciones de OEFA con coordenadas\n(líneas: a menos de 15 km)", loc="left")
ax.set_aspect(1 / np.cos(np.radians(np.nanmean(con_coord["lat"]) if len(con_coord) else -10)))
guardar(fig, "eda_mapa_estaciones.png")

## 5. Rango de fechas y años completos

Un **año completo** es un año calendario con PM (PM2.5 o PM10) en al menos el 75 % de sus horas
(umbral `UMBRAL_ANIO`). Decide Q1 (estacionalidad), Q3 y Q4.

In [ ]:
UMBRAL_ANIO = 0.75
COLS_PM = [c for c in ["pm2_5", "pm10"] if c in ancho]
ancho["_hay_pm"] = ancho[COLS_PM].notna().any(axis=1) if COLS_PM else False
ancho["_anio"] = ancho["fecha_hora"].dt.year

rango = ancho.groupby("estacion").agg(inicio=("fecha_hora", "min"), fin=("fecha_hora", "max"),
                                       filas=("fecha_hora", "size"), horas_con_pm=("_hay_pm", "sum"))
rango["horas_rango"] = ((rango["fin"] - rango["inicio"]) / pd.Timedelta(hours=1)).astype(int) + 1
pm_anio = ancho[ancho["_hay_pm"]].groupby(["estacion", "_anio"]).size().rename("horas").reset_index()
pm_anio["horas_del_anio"] = [8784 if (a % 4 == 0 and (a % 100 != 0 or a % 400 == 0)) else 8760 for a in pm_anio["_anio"]]
pm_anio["cobertura"] = pm_anio["horas"] / pm_anio["horas_del_anio"]
completos = pm_anio[pm_anio["cobertura"] >= UMBRAL_ANIO].groupby("estacion")["_anio"].agg(list)
rango["anios_completos"] = [completos.get(e, []) for e in rango.index]
rango["n_anios_completos"] = rango["anios_completos"].str.len()
rango["cobertura_por_anio"] = [", ".join(f"{int(a)}: {100 * c:.0f} %" for a, c in
                                         pm_anio.loc[pm_anio["estacion"] == e, ["_anio", "cobertura"]].to_numpy())
                               for e in rango.index]
display(rango)
print(f"Rango global: {ancho['fecha_hora'].min()} a {ancho['fecha_hora'].max()}")
print(f"Estaciones con al menos un año completo de PM: {(rango['n_anios_completos'] > 0).sum()} de {N_EST}")

## 6. Completitud estación × parámetro

Fracción de las horas del rango de cada estación (de su primera a su última fila) con dato válido
del parámetro, después de la limpieza. Decide Q2 (ambas PM), Q6 (gases), Q4 (comparabilidad) y las
features del K-Means: se prefiere **menos features con más estaciones** (spec PC3, §4.3).

In [ ]:
comp = ancho.groupby("estacion")[PARAMS].count().div(rango["horas_rango"], axis=0)
comp["pm_alguna"] = ancho.groupby("estacion")["_hay_pm"].sum() / rango["horas_rango"]
comp["pm_ambas"] = (ancho[["pm2_5", "pm10"]].notna().all(axis=1).groupby(ancho["estacion"]).sum() / rango["horas_rango"]
                    if {"pm2_5", "pm10"} <= set(PARAMS) else 0.0)
for n, f in CONJUNTOS.items():
    comp[n.split(" ")[0]] = regla8.drop(index="TOTAL")[n] / rango["horas_rango"]
comp = comp.reindex(ESTACIONES)
comp.round(4).to_csv(REPORTS / "eda_completitud.csv")
display((100 * comp).round(1))

In [ ]:
cols_fig = [c for c in al.CANONICOS if c in comp] + ["pm_alguna", "pm_ambas"] + [n.split(" ")[0] for n in CONJUNTOS]
M = 100 * comp[cols_fig].to_numpy(dtype=float)
alto = max(3.0, 0.32 * len(ESTACIONES) + 1.6)
fig, ax = plt.subplots(figsize=(max(8, 0.62 * len(cols_fig) + 3), alto))
im = ax.imshow(M, cmap="aire_seq", vmin=0, vmax=100, aspect="auto")
ETIQ_COMP = {"pm_alguna": "PM (alguna)", "pm_ambas": "PM (las dos)"}
ax.set_xticks(range(len(cols_fig)), [ETIQ_COMP.get(c, al.NOMBRE_TXT.get(c, c)) for c in cols_fig], rotation=45, ha="right")
ax.set_yticks(range(len(ESTACIONES)), ESTACIONES)
ax.grid(False)
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        if not np.isnan(M[i, j]):
            ax.text(j, i, f"{M[i, j]:.0f}", ha="center", va="center", fontsize=7,
                    color="white" if M[i, j] > 55 else al.TINTA)
ax.axvline(len([c for c in al.CANONICOS if c in comp]) - 0.5, color=al.TINTA_2, linewidth=1)
fig.colorbar(im, ax=ax, label="% de las horas del rango con dato", shrink=0.8)
ax.set_title("Completitud por estación y parámetro, después de la limpieza", loc="left")
al.nota_al_pie(fig, f"Fracción de las horas del rango de cada estación con dato válido. Bronze: {BRONZE.name}. "
               "F1/F2/M0–M2: filas completas para cada conjunto de features de la calibración.")
guardar(fig, "completitud.png", informe=True)

In [ ]:
# ¿Qué parámetros pueden ser features? Cuántas estaciones los tienen con >= 50 % y >= 75 % de horas.
resumen_feat = pd.DataFrame({
    "estaciones_>=50%": (comp[PARAMS] >= 0.5).sum(),
    "estaciones_>=75%": (comp[PARAMS] >= 0.75).sum(),
    "mediana_%": (100 * comp[PARAMS].median()).round(1),
}).reindex([c for c in al.CANONICOS if c in PARAMS])
resumen_feat["de"] = N_EST
display(resumen_feat)
GRAN_MAYORIA = 0.75  # spec de calibración §2.5: los gases entran si los reporta la gran mayoría
gases_entran = [g for g in al.GASES if g in resumen_feat.index and resumen_feat.loc[g, "estaciones_>=50%"] >= GRAN_MAYORIA * N_EST]
print("Gases que podrían entrar al modelo (>= 50 % de horas en >= 75 % de las estaciones):", gases_entran or "ninguno")

## 7. Huecos

Un **hueco** es un tramo de horas sin PM2.5 ni PM10 entre la primera y la última hora con PM de la
estación. Se mide su longitud y, para Q9, el PM de las **6 horas anteriores** a cada hueco de al
menos `HUECO_MIN_H` horas frente al resto: si las estaciones dejan de reportar cuando el aire
empeora, el dato subestima el problema. El PM se expresa relativo a la media de cada estación (así
se pueden juntar estaciones de niveles distintos) y el intervalo es un bootstrap por huecos.

In [ ]:
HUECO_MIN_H = 3
VENTANA_PREVIA_H = 6

hu = al.huecos(ancho, COLS_PM)
rango_pm = ancho[ancho["_hay_pm"]].groupby("estacion")["fecha_hora"].agg(["min", "max"])
horas_rango_pm = ((rango_pm["max"] - rango_pm["min"]) / pd.Timedelta(hours=1)).astype(int) + 1
por_est_h = (hu.groupby("estacion")["horas"]
             .agg(n_huecos="size", horas_en_huecos="sum", mediana_h="median",
                  p90_h=lambda s: s.quantile(0.9), maximo_h="max")
             .reindex(ESTACIONES).fillna(0))
por_est_h["frac_en_huecos"] = por_est_h["horas_en_huecos"] / horas_rango_pm.reindex(ESTACIONES)
por_est_h["frac_en_huecos_>24h"] = (hu[hu["horas"] > 24].groupby("estacion")["horas"].sum()
                                     .reindex(ESTACIONES).fillna(0) / horas_rango_pm.reindex(ESTACIONES))
display(por_est_h.round(3))

cortes = [0, 1, 6, 24, 168, np.inf]
etiquetas = ["1 h", "2–6 h", "7–24 h", "1–7 días", "> 7 días"]
hu["tramo"] = pd.cut(hu["horas"], cortes, labels=etiquetas)
dist_huecos = hu.groupby("tramo", observed=False)["horas"].agg(huecos="size", horas="sum")
dist_huecos["% de horas perdidas"] = 100 * dist_huecos["horas"] / max(dist_huecos["horas"].sum(), 1)
display(dist_huecos)

In [ ]:
rej_pm = al.regularizar(ancho, COLS_PM)
filas_q9, pre_todos, resto_todos = [], [], []
for est, g in rej_pm.groupby("estacion"):
    cols = [c for c in COLS_PM if g[c].notna().any()]
    if not cols:
        continue
    col = max(cols, key=lambda c: g[c].notna().sum())  # el PM con más datos en la estación
    v = g[col].to_numpy()
    rel = v / np.nanmean(v)
    tiempos = g["fecha_hora"].to_numpy()
    previa = np.zeros(len(g), dtype=bool)
    medias_previas, exc_previas = [], []
    for ini in hu.loc[(hu["estacion"] == est) & (hu["horas"] >= HUECO_MIN_H), "inicio"]:
        pos = np.searchsorted(tiempos, np.datetime64(ini))
        a = max(0, pos - VENTANA_PREVIA_H)
        previa[a:pos] = True
        tramo = rel[a:pos]
        if np.isfinite(tramo).any():
            medias_previas.append(np.nanmean(tramo))
            exc_previas.append(np.nanmean(v[a:pos][np.isfinite(v[a:pos])] > al.ECA[col]))
    resto = rel[~previa & np.isfinite(rel)]
    r, lo, hi = al.ic_bootstrap_cociente(np.array(medias_previas), resto, semilla=1)
    exc_resto = np.nanmean(v[~previa & np.isfinite(v)] > al.ECA[col])
    filas_q9.append({"estacion": est, "pm": col, "huecos_>=min": len(medias_previas),
                     "pm_previo/resto": r, "ic95_bajo": lo, "ic95_alto": hi,
                     "%_sobre_eca_previo": 100 * np.mean(exc_previas) if exc_previas else np.nan,
                     "%_sobre_eca_resto": 100 * exc_resto})
    pre_todos += medias_previas
    resto_todos.append(resto)
q9 = pd.DataFrame(filas_q9).set_index("estacion")
q9_global = al.ic_bootstrap_cociente(np.array(pre_todos), np.concatenate(resto_todos) if resto_todos else np.array([]), semilla=2)
display(q9.round(3))
print(f"Todas las estaciones: PM previo / resto = {q9_global[0]:.3f} (IC 95 % {q9_global[1]:.3f}–{q9_global[2]:.3f}), "
      f"{len(pre_todos)} huecos de >= {HUECO_MIN_H} h")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
axs[0].bar(dist_huecos.index.astype(str), dist_huecos["huecos"], color=al.PALETA[0], width=0.6)
axs[0].set_title("Huecos por longitud", loc="left")
axs[0].set_ylabel("Número de huecos")
axs[1].bar(dist_huecos.index.astype(str), dist_huecos["% de horas perdidas"], color=al.PALETA[0], width=0.6)
axs[1].set_title("Horas sin PM por longitud del hueco", loc="left")
axs[1].set_ylabel("% de las horas perdidas")
for ax in axs:
    ax.grid(axis="x", visible=False)
guardar(fig, "eda_huecos.png")

## 8. Episodios frente al ECA

Frecuencia de horas con PM2.5 > 50 o PM10 > 100 µg/m³ (lectura horaria) y del **promedio móvil de
24 h** (ventana que termina en la hora, al menos 18 horas con dato). El ECA es un promedio de 24 h:
una hora sobre el umbral no es por sí sola un incumplimiento. Un **episodio** es un tramo continuo
de horas con el promedio móvil sobre el ECA. Si la frecuencia es menor que 0,5 % en todas las
estaciones, no hay episodios que clusterizar (Q3, Q4).

In [ ]:
movil = al.movil_24h(rej_pm)
movil["nuevo"] = movil["excede_24h"] & ~movil.groupby("estacion")["excede_24h"].shift(1, fill_value=False)
eca = pd.DataFrame({
    "horas_con_pm": ancho.groupby("estacion")["_hay_pm"].sum(),
    "%_horas_sobre_eca": 100 * al.excede_horario(ancho).groupby(ancho["estacion"]).sum()
                         / ancho.groupby("estacion")["_hay_pm"].sum().replace(0, np.nan),
    "%_24h_sobre_eca": 100 * movil.groupby("estacion")["excede_24h"].sum()
                       / movil.groupby("estacion")["hay_24h"].sum().replace(0, np.nan),
    "episodios_24h": movil.groupby("estacion")["nuevo"].sum(),
}).reindex(ESTACIONES)
for p in ["pm2_5", "pm10"]:
    if p in ancho:
        eca[f"%_horas_{p}_sobre"] = 100 * (ancho[p] > al.ECA[p]).groupby(ancho["estacion"]).sum() / ancho[p].notna().groupby(ancho["estacion"]).sum().replace(0, np.nan)
display(eca.round(2))
print(f"Episodios de 24 h en total: {int(eca['episodios_24h'].sum())}")

In [ ]:
orden = eca["%_24h_sobre_eca"].fillna(-1).sort_values().index
fig, ax = plt.subplots(figsize=(8, max(3, 0.3 * len(orden) + 1.2)))
y = np.arange(len(orden))
ax.barh(y + 0.2, eca.loc[orden, "%_horas_sobre_eca"], height=0.38, color=al.PALETA[0], label="Lectura horaria")
ax.barh(y - 0.2, eca.loc[orden, "%_24h_sobre_eca"], height=0.38, color=al.PALETA[1], label="Promedio móvil de 24 h")
ax.axvline(0.5, color=al.CRITICO, linewidth=1, linestyle="--")
ax.text(0.5, len(orden) - 0.4, " 0,5 %", color=al.CRITICO, fontsize=8, va="bottom")
ax.set_yticks(y, orden)
ax.set_xlabel("% de horas sobre el ECA (PM2.5 > 50 o PM10 > 100 µg/m³)")
ax.set_title("Horas sobre el ECA por estación", loc="left")
ax.grid(axis="y", visible=False)
ax.legend(loc="lower right")
guardar(fig, "eda_eca.png")

## 9. PM2.5 ≤ PM10 y fracción fina

PM2.5 está contenida en PM10: una hora con PM2.5 > PM10 es error de medición. La fracción fina
PM2.5/PM10 separa **combustión** (alta) de **polvo** (baja); para Q2 hace falta que muchas
estaciones midan las dos a la vez.

In [ ]:
if {"pm2_5", "pm10"} <= set(PARAMS):
    ambas = ancho[["pm2_5", "pm10"]].notna().all(axis=1) & (ancho["pm10"] > 0)
    razon = (ancho["pm2_5"] / ancho["pm10"]).where(ambas)
    ff = pd.DataFrame({
        "horas_con_pm": ancho.groupby("estacion")["_hay_pm"].sum(),
        "horas_con_ambas": ambas.groupby(ancho["estacion"]).sum(),
    }).reindex(ESTACIONES)
    ff["simultaneidad"] = ff["horas_con_ambas"] / ff["horas_con_pm"].replace(0, np.nan)
    ff["%_pm25_mayor_pm10"] = 100 * (razon > 1).groupby(ancho["estacion"]).sum().reindex(ESTACIONES) / ff["horas_con_ambas"].replace(0, np.nan)
    q = razon.clip(0, 1).groupby(ancho["estacion"]).quantile([0.25, 0.5, 0.75]).unstack()
    ff[["fraccion_p25", "fraccion_mediana", "fraccion_p75"]] = q.reindex(ESTACIONES).to_numpy()
    violacion_global = float((razon > 1).sum() / max(ambas.sum(), 1))
    display(ff.round(3))
    print(f"PM2.5 > PM10 en {100 * violacion_global:.2f} % de las horas con ambas")

    datos = [razon[ancho["estacion"] == e].dropna().clip(0, 1.5).to_numpy() for e in ESTACIONES]
    con = [i for i, d in enumerate(datos) if len(d)]
    fig, ax = plt.subplots(figsize=(8, max(3, 0.3 * len(con) + 1.2)))
    al.boxplot_horizontal(ax, [datos[i] for i in con], widths=0.55, showfliers=False,
               medianprops={"color": al.PALETA[1], "linewidth": 2}, boxprops={"color": al.TINTA_2},
               whiskerprops={"color": al.TINTA_2}, capprops={"color": al.TINTA_2})
    ax.set_yticks(range(1, len(con) + 1), [ESTACIONES[i] for i in con])
    ax.axvline(1, color=al.CRITICO, linewidth=1, linestyle="--")
    ax.text(1, len(con) + 0.5, " PM2.5 = PM10", color=al.CRITICO, fontsize=8)
    ax.set_xlabel("PM2.5 / PM10 (sin recortar; > 1 es error de medición)")
    ax.set_title("Fracción fina por estación", loc="left")
    ax.grid(axis="y", visible=False)
    guardar(fig, "eda_fraccion_fina.png")
else:
    ff = pd.DataFrame(index=ESTACIONES, data={"simultaneidad": 0.0, "%_pm25_mayor_pm10": np.nan})
    violacion_global = float("nan")
    print("No hay PM2.5 y PM10 a la vez en el silver: Q2 no se puede medir.")

In [ ]:
# Figura del informe: distribución de PM2.5, PM10 (escala log, con el ECA) y de la fracción fina, por estación.
paneles = [(p, f"{al.NOMBRE_TXT[p]} (µg/m³, escala log)") for p in ["pm2_5", "pm10"] if p in ancho]
if {"pm2_5", "pm10"} <= set(ancho):
    paneles.append(("fraccion", "Fracción fina PM2.5/PM10 (sin recortar)"))
fig, axs = plt.subplots(1, len(paneles), figsize=(4.6 * len(paneles) + 1.5, 0.34 * N_EST + 1.8), sharey=True, squeeze=False)
for ax, (p, titulo) in zip(axs[0], paneles):
    if p == "fraccion":
        serie = (ancho["pm2_5"] / ancho["pm10"].where(ancho["pm10"] > 0))
    else:
        serie = ancho[p].where(ancho[p] > 0)
    datos = [serie[ancho["estacion"] == e].dropna().to_numpy() for e in ESTACIONES]
    con = [i for i, d in enumerate(datos) if len(d)]
    al.boxplot_horizontal(ax, [datos[i] if i in con else [np.nan] for i in range(N_EST)], widths=0.55, showfliers=False,
                          medianprops={"color": al.PALETA[0], "linewidth": 2}, boxprops={"color": al.TINTA_2},
                          whiskerprops={"color": al.TINTA_2}, capprops={"color": al.TINTA_2})
    if p == "fraccion":
        ax.axvline(1, color=al.CRITICO, linewidth=1, linestyle="--")
        ax.text(1, N_EST + 0.6, " PM2.5 = PM10", color=al.CRITICO, fontsize=8)
    else:
        ax.set_xscale("log")
        lo_, hi_ = np.nanpercentile(serie, 0.5), np.nanpercentile(serie, 99.5)
        ticks = [t for t in [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, 2000] if lo_ / 1.5 <= t <= max(hi_, al.ECA[p]) * 1.5]
        ax.xaxis.set_major_locator(plt.FixedLocator(ticks))
        ax.xaxis.set_minor_locator(plt.NullLocator())
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:g}"))
        ax.axvline(al.ECA[p], color=al.CRITICO, linewidth=1.1, linestyle="--")
        ax.text(al.ECA[p], N_EST + 0.6, f" ECA 24 h = {al.ECA[p]:.0f}", color=al.CRITICO, fontsize=8)
    ax.set_title(titulo, loc="left", fontsize=10)
    ax.grid(axis="y", visible=False)
axs[0][0].set_yticks(range(1, N_EST + 1), ESTACIONES, fontsize=8)
fig.suptitle("Distribución del material particulado por estación", x=0.01, ha="left", fontweight="bold", fontsize=13)
fig.tight_layout()
al.nota_al_pie(fig, "Cajas: cuartiles; bigotes: 1,5 × rango intercuartil; sin atípicos. El ECA es un promedio de 24 h: "
               "una lectura horaria sobre la línea no es por sí sola un incumplimiento.")
guardar(fig, "distribuciones.png", informe=True)

## 10. Ciclos diario y semanal

Perfil medio por hora del día y por día de la semana. η² es la fracción de la varianza (de
log(1 + PM)) que explica la hora: si la hora lo explica casi todo, Q1 es trivial («de noche»); para
los gases, un ciclo marcado por hora o por día de semana es la firma de una fuente industrial (Q6).

In [ ]:
ancho["_hora"] = ancho["fecha_hora"].dt.hour
ancho["_dow"] = ancho["fecha_hora"].dt.day_of_week
VARS_CICLO = [c for c in ["pm10", "pm2_5"] + al.GASES if c in ancho]
ciclo = []
for est, g in ancho.groupby("estacion"):
    fila = {"estacion": est}
    for c in VARS_CICLO:
        if g[c].notna().sum() >= 48:
            lv = np.log1p(g[c].clip(lower=0))
            fila[f"eta2_hora_{c}"] = al.eta2(lv, g["_hora"])
            fila[f"eta2_semana_{c}"] = al.eta2(lv, g["_dow"])
    ciclo.append(fila)
ciclo = pd.DataFrame(ciclo).set_index("estacion").reindex(ESTACIONES)
display(ciclo.round(3))
cols_pm_eta = [c for c in ciclo.columns if c.startswith("eta2_hora_pm")]
eta2_pm = ciclo[cols_pm_eta].max(axis=1) if cols_pm_eta else pd.Series(np.nan, index=ESTACIONES)
print(f"η² de la hora sobre log PM: mediana {eta2_pm.median():.3f}, máximo {eta2_pm.max():.3f}")

In [ ]:
n = len(ESTACIONES)
ncol = min(6, n)
nfil = math.ceil(n / ncol)
for nombre_fig, clave, etiq_x, ticks in [("eda_ciclo_diario.png", "_hora", "Hora del día", range(0, 24, 6)),
                                          ("eda_ciclo_semanal.png", "_dow", "Día de la semana (0 = lunes)", range(7))]:
    fig, axs = plt.subplots(nfil, ncol, figsize=(2.3 * ncol + 1, 1.9 * nfil + 1), sharex=True, squeeze=False)
    for ax, est in zip(axs.flat, ESTACIONES):
        g = ancho[ancho["estacion"] == est]
        for j, c in enumerate([c for c in ["pm10", "pm2_5"] if c in g]):
            perfil = g.groupby(clave)[c].mean()
            if perfil.notna().any():
                ax.plot(perfil.index, perfil.to_numpy(), color=al.PALETA[j], label=al.NOMBRE_TXT[c])
        ax.set_title(est, fontsize=8, loc="left")
        ax.set_xticks(list(ticks))
        ax.tick_params(labelsize=7)
    for ax in list(axs.flat)[n:]:
        ax.axis("off")
    axs.flat[0].legend(fontsize=7)
    fig.supxlabel(etiq_x, fontsize=9)
    fig.supylabel("Media (µg/m³)", fontsize=9)
    fig.suptitle(f"Ciclo {'diario' if clave == '_hora' else 'semanal'} de PM por estación", x=0.01, ha="left",
                 fontweight="bold")
    fig.tight_layout()
    guardar(fig, nombre_fig)

## 11. Meteorología junto con PM

Qué estaciones tienen viento y humedad (features M1 del K-Means) en las mismas horas que el PM, y
temperatura y presión (M2).

In [ ]:
def junto(cols):
    if not all(c in ancho for c in cols):
        return pd.Series(0.0, index=ESTACIONES)
    m = ancho["_hay_pm"] & ancho[cols].notna().all(axis=1)
    return (m.groupby(ancho["estacion"]).sum() / ancho.groupby("estacion")["_hay_pm"].sum().replace(0, np.nan)).reindex(ESTACIONES)

meteo = pd.DataFrame({"viento_y_humedad": junto(["viento_velocidad", "humedad"]),
                      "temperatura_y_presion": junto(["temperatura", "presion"]),
                      "las_cuatro": junto(["viento_velocidad", "humedad", "temperatura", "presion"])})
display((100 * meteo).round(1))

## 12. La criba: veredicto por candidata

Cada candidata tiene un criterio explícito sobre las mediciones de arriba (§2 de la ficha de
preguntas). Los umbrales son puntos de partida y están todos en `U`: si se ajustan, el informe dice
cuáles se usaron. *Pasa* = el dato permite la pregunta; *no pasa* = la mata lo que la ficha dice que
la mata; *dudoso* = hay datos pero no alcanzan, o falta algo externo.

In [ ]:
U = {
    "estaciones_min": max(3, math.ceil(0.25 * N_EST)),  # «varias estaciones»
    "completitud_buena": 0.75,      # fracción de horas del rango con el dato
    "meteo_con_pm": 0.75,           # fracción de horas con PM que también tienen viento y humedad
    "eca_min": 0.5,                 # % de horas con el promedio de 24 h sobre el ECA
    "episodios_min": 30,            # episodios de 24 h en total para estimar proporciones
    "huecos_largos_max": 0.10,      # mediana entre estaciones de la fracción de horas en huecos > 24 h
    "eta2_trivial": 0.5,            # si la hora explica más que esto de la varianza de log PM, Q1 es trivial
    "simultaneidad_min": 0.5,       # horas con ambas PM / horas con alguna
    "violacion_max": 0.05, "violacion_dudosa": 0.15,
    "solape_dias_min": 90,          # días con PM en las dos estaciones de una pareja cercana
    "gas_completitud": 0.5, "eta2_firma_hora": 0.05, "eta2_firma_semana": 0.05,
    "q9_huecos_min": 20, "q9_cociente_min": 1.10, "q9_frac_huecos_min": 0.01,
    "q9_huecos_min_estacion": 10,   # huecos para leer el efecto en una estación sola
}
VEREDICTO = {True: "pasa", False: "no pasa", None: "dudoso"}


def v(pasa, no_pasa):
    return "pasa" if pasa else ("no pasa" if no_pasa else "dudoso")


criba = {}
pm_ok = comp["pm_alguna"] >= U["completitud_buena"]
met_ok = meteo["viento_y_humedad"] >= U["meteo_con_pm"]
anio_ok = rango["n_anios_completos"].reindex(ESTACIONES) > 0
eca_ok = eca["%_24h_sobre_eca"] >= U["eca_min"]
trivial = bool(eta2_pm.median() >= U["eta2_trivial"]) if eta2_pm.notna().any() else False

# Q1
n1, n1b = int((pm_ok & met_ok & anio_ok).sum()), int((pm_ok & met_ok).sum())
criba["Q1"] = dict(
    veredicto=v(n1 >= U["estaciones_min"] and eca_ok.any() and not trivial, n1b == 0 or not eca_ok.any()),
    cifra=(f"{n1} de {N_EST} estaciones con PM y viento+humedad en >= {al.pct(U['completitud_buena'])} de las horas "
           f"y al menos un año completo ({n1b} sin exigir el año); {int(eca_ok.sum())} estaciones con >= {U['eca_min']:.1f} % "
           f"de horas de 24 h sobre el ECA; la hora explica en mediana {100 * eta2_pm.median():.0f} % de la varianza de log PM"
           + (" (trivial: el ciclo horario lo explica casi todo)" if trivial else "")),
    criterio=f"pasa con >= {U['estaciones_min']} estaciones con PM, meteo y un año completo, episodios y η² hora < {U['eta2_trivial']}")

# Q2
n2 = int((ff["simultaneidad"] >= U["simultaneidad_min"]).sum())
viol = violacion_global
criba["Q2"] = dict(
    veredicto=v(n2 >= U["estaciones_min"] and viol <= U["violacion_max"],
                n2 == 0 or not (viol <= U["violacion_dudosa"])),
    cifra=(f"{n2} de {N_EST} estaciones miden PM2.5 y PM10 a la vez en >= {al.pct(U['simultaneidad_min'])} de sus horas con PM; "
           f"PM2.5 > PM10 en {100 * viol:.2f} % de las horas con ambas"),
    criterio=f"pasa con >= {U['estaciones_min']} estaciones simultáneas y violaciones <= {al.pct(U['violacion_max'])}")

# Q3
n_eca = int(eca_ok.sum())
episodios = int(eca["episodios_24h"].sum())
largos = float(por_est_h["frac_en_huecos_>24h"].median())
criba["Q3"] = dict(
    veredicto=v(n_eca >= 1 and episodios >= U["episodios_min"] and largos <= U["huecos_largos_max"],
                n_eca == 0),
    cifra=(f"{n_eca} estaciones con >= {U['eca_min']:.1f} % de horas de 24 h sobre el ECA (máximo "
           f"{eca['%_24h_sobre_eca'].max():.2f} %); {episodios} episodios de 24 h; en mediana "
           f"{100 * largos:.1f} % de las horas caen en huecos de más de 24 h"),
    criterio=(f"no pasa si ninguna estación llega a {U['eca_min']:.1f} %; pasa con >= {U['episodios_min']} episodios "
              f"y huecos largos <= {al.pct(U['huecos_largos_max'])}; revisar con Q9"))

# Q4
con_pm = comp["pm_alguna"] > 0
frac_buenas = float((comp.loc[con_pm, "pm_alguna"] >= U["completitud_buena"]).mean()) if con_pm.any() else 0.0
spread = float(comp.loc[con_pm, "pm_alguna"].quantile(0.9) - comp.loc[con_pm, "pm_alguna"].quantile(0.1)) if con_pm.any() else np.nan
criba["Q4"] = dict(
    veredicto=v(frac_buenas >= 0.75 and spread <= 0.20, frac_buenas < 0.5),
    cifra=(f"{100 * frac_buenas:.0f} % de las estaciones con PM tienen PM en >= {al.pct(U['completitud_buena'])} de sus horas; "
           f"diferencia p90–p10 de completitud entre estaciones: {100 * spread:.0f} puntos"),
    criterio="pasa si >= 75 % de las estaciones tienen completitud buena y la dispersión p90–p10 <= 20 puntos")

# Q5
if len(parejas):
    buenas = parejas[parejas["dias_con_pm_en_ambas"] >= U["solape_dias_min"]]
    n5 = len(buenas)
    criba["Q5"] = dict(
        veredicto=v(n5 >= 2 or any(len(g) >= 3 for g in grupos15), False),
        cifra=(f"{len(parejas)} parejas de estaciones a menos de 15 km ({n5} con >= {U['solape_dias_min']} días de PM "
               f"en ambas); grupos: {grupos15}"),
        criterio=f"pasa con >= 2 parejas a < 15 km y >= {U['solape_dias_min']} días de solape; no pasa sin parejas")
else:
    criba["Q5"] = dict(veredicto="no pasa", cifra="ninguna pareja de estaciones a menos de 15 km",
                       criterio="no pasa sin parejas cercanas")

# Q6
gases_col = [g for g in al.GASES if g in comp]
con_gas = [e for e in ESTACIONES if any(comp.loc[e, g] >= U["gas_completitud"] for g in gases_col)]
firma = {}
for e in con_gas:
    for g in gases_col:
        if comp.loc[e, g] >= U["gas_completitud"]:
            eh, es = ciclo.loc[e].get(f"eta2_hora_{g}", np.nan), ciclo.loc[e].get(f"eta2_semana_{g}", np.nan)
            if (eh >= U["eta2_firma_hora"]) or (es >= U["eta2_firma_semana"]):
                if e not in firma or max(eh, es) > firma[e][1]:
                    firma[e] = (g, max(eh, es), eh, es)
firma = [f"{e} ({g}, η² hora {eh:.2f}, semana {es:.2f})" for e, (g, _, eh, es) in sorted(firma.items())]
criba["Q6"] = dict(
    veredicto=v(len(firma) >= 2, len(con_gas) == 0),
    cifra=(f"{len(con_gas)} de {N_EST} estaciones con algún gas en >= {al.pct(U['gas_completitud'])} de sus horas "
           f"(gases medidos: {gases_col or 'ninguno'}); con firma horaria o semanal: {firma or 'ninguna'}"),
    criterio=(f"pasa con >= 2 estaciones con gas y η² hora >= {U['eta2_firma_hora']} o semana >= {U['eta2_firma_semana']}; "
              "no pasa sin gases"))

# Q7
n_coord = int(estaciones["lat"].notna().sum())
regiones_cub = sorted(set(r for r in estaciones["region"].dropna()))
criba["Q7"] = dict(
    veredicto="pasa" if n_coord >= 0.9 * len(estaciones) else "dudoso",
    cifra=(f"{n_coord} de {len(estaciones)} estaciones con coordenadas; cubren {len(regiones_cub)} de las 25 regiones "
           f"({', '.join(al.NOMBRE_REGION.get(r, r) for r in regiones_cub)}). La población por distrito (INEI) es externa"),
    criterio="siempre entra, como advertencia: una zona sin estación no es una zona con aire limpio")

# Q8
lima = [e for e in estaciones.index if estaciones.loc[e, "region"] in ("LIMA", "CALLAO")]
criba["Q8"] = dict(
    veredicto="no pasa" if not lima else "dudoso",
    cifra=f"{len(lima)} estaciones de OEFA en Lima o Callao ({', '.join(lima) or 'ninguna'}); falta SENAMHI (va al TB2)",
    criterio="necesita las estaciones de SENAMHI a pocos km: se decide en el TB2")

# Q9
n_h9 = len(pre_todos)
frac_hu = float(por_est_h["frac_en_huecos"].median())
est_efecto = (list(q9.index[(q9["ic95_bajo"] > U["q9_cociente_min"]) & (q9["huecos_>=min"] >= U["q9_huecos_min_estacion"])])
              if len(q9) else [])
pocos = n_h9 < U["q9_huecos_min"] or frac_hu < U["q9_frac_huecos_min"]
efecto = (q9_global[1] > U["q9_cociente_min"]) or len(est_efecto) >= 2
aleatorios = not (q9_global[1] > 1) and not est_efecto  # el PM previo no se distingue del resto
criba["Q9"] = dict(
    veredicto=v(efecto and not pocos, aleatorios),
    cifra=(f"{n_h9} huecos de >= {HUECO_MIN_H} h; mediana de horas en huecos {100 * frac_hu:.1f} %; PM de las "
           f"{VENTANA_PREVIA_H} h previas / resto = {q9_global[0]:.2f} (IC 95 % {q9_global[1]:.2f}–{q9_global[2]:.2f}); "
           f"estaciones con efecto claro (IC > {U['q9_cociente_min']}): {est_efecto or 'ninguna'}"),
    criterio=(f"pasa si hay >= {U['q9_huecos_min']} huecos y >= {al.pct(U['q9_frac_huecos_min'])} de horas en huecos, y el PM "
              f"previo supera al resto (IC > {U['q9_cociente_min']}) en conjunto o en >= 2 estaciones con >= "
              f"{U['q9_huecos_min_estacion']} huecos; no pasa si el PM previo no se distingue del resto (huecos aleatorios); "
              "dudoso si hay efecto pero los huecos son pocos (entonces es una línea del informe, no una pregunta)"))
if criba["Q9"]["veredicto"] == "pasa" and criba["Q3"]["veredicto"] == "pasa":
    criba["Q3"]["veredicto"] = "dudoso"
    criba["Q3"]["cifra"] += "; Q9 pasa: los huecos coinciden con los episodios y sesgan la anticipación"

TITULOS = {
    "Q1": "¿Cuándo y dónde se concentran los episodios sobre el ECA, y con qué meteorología llegan?",
    "Q2": "¿De qué está hecho cada episodio: combustión o polvo?",
    "Q3": "¿Con cuánta anticipación se puede avisar?",
    "Q4": "¿Cuánto del año respira fuera del ECA la población cercana a cada estación?",
    "Q5": "¿Los episodios son regionales o locales?",
    "Q6": "¿Los gases tienen la firma horaria de una fuente industrial?",
    "Q7": "¿Qué parte del país tiene una estación cerca?",
    "Q8": "¿Las estaciones de OEFA ven lo mismo que las de SENAMHI en Lima?",
    "Q9": "¿Los huecos del dato coinciden con los episodios?",
}
resumen_criba = pd.DataFrame([{"Q": q, "pregunta": TITULOS[q], "veredicto": c["veredicto"], "cifra": c["cifra"]}
                              for q, c in criba.items()])
display(resumen_criba[["Q", "veredicto", "cifra"]])

## 13. `reports/eda_criba.md`

In [ ]:
def sugerencia():
    ver = {q: c["veredicto"] for q, c in criba.items()}
    base = "Q3 apoyada en Q1" if ver["Q3"] != "no pasa" and ver["Q1"] != "no pasa" else (
        "Q1 sola (Q3 no pasa)" if ver["Q1"] != "no pasa" else "ninguna de las dos pasa: replantear con Q4")
    dif = [q for q in ("Q5", "Q2") if ver[q] == "pasa"] or [q for q in ("Q5", "Q2") if ver[q] == "dudoso"]
    return base, (dif[0] if dif else "ni Q5 ni Q2: el diferencial queda en Q9/Q7")

base, dif = sugerencia()
hoy = dt.datetime.now().strftime("%Y-%m-%d %H:%M")
L = [
    "# Criba de preguntas candidatas — EDA del bronze de OEFA",
    "",
    f"> Generado por `notebooks/01_eda_criba.ipynb` el {hoy}. Bronze: `{info_bronze['archivo']}` "
    f"({al.miles(info_bronze['bytes'])} bytes, sha256 `{info_bronze['sha256']}`). Config de limpieza: {origen_cfg}.",
]
if SINTETICO:
    L += ["", "> **DATOS SINTÉTICOS** (muestra de `data/sample/`, `notebooks/sintetico.py` o estaciones SINT-/SIN-): "
          "las cifras prueban el notebook, no dicen nada del aire del Perú."]
L += [
    "",
    "Criterios y umbrales en la celda `U` del notebook; ficha de las candidatas en "
    "`docs/specs/2026-10-31-preguntas-candidatas.md`.",
    "",
    "## Resumen",
    "",
    al.tabla_md(resumen_criba[["Q", "veredicto", "cifra"]]),
    "",
    f"**Lectura automática (a validar el martes con el equipo)**: base = {base}; diferencial = {dif}; Q7 y Q9 "
    "entran como advertencias. Lo que no pasó va al informe como «preguntas que este dato no puede responder».",
    "",
    f"Datos: {al.miles(fl['leidas'])} filas largas leídas, {al.miles(fl['procesadas'])} válidas, {al.miles(len(ancho))} filas anchas, "
    f"{N_EST} estaciones con datos válidos, de {ancho['fecha_hora'].min():%Y-%m-%d} a {ancho['fecha_hora'].max():%Y-%m-%d}.",
]
for q, c in criba.items():
    L += ["", f"## {q}. {TITULOS[q]}", "", f"**Veredicto: {c['veredicto']}.** {c['cifra']}.", "",
          f"Criterio: {c['criterio']}."]
L += ["", "## Mediciones", "", "### Rango de fechas y años completos", "",
      al.tabla_md(rango[["inicio", "fin", "horas_rango", "horas_con_pm", "n_anios_completos", "cobertura_por_anio"]]
                  .assign(inicio=lambda d: d["inicio"].dt.strftime("%Y-%m-%d"), fin=lambda d: d["fin"].dt.strftime("%Y-%m-%d")),
                  indice=True),
      "", "### Completitud estación × parámetro (% de las horas del rango)", "",
      "Tabla para decidir las features: se prefiere menos features con más estaciones.", "",
      al.tabla_md((100 * comp).round(1), decimales=1, indice=True),
      "", f"Gases que podrían entrar al modelo (>= 50 % de horas en >= 75 % de las estaciones): {gases_entran or 'ninguno'}.",
      "", "### Filas completas por estación (regla 8) para los conjuntos de la calibración", "",
      al.tabla_md(regla8, decimales=0, indice=True),
      "", "### Huecos por estación", "", al.tabla_md(por_est_h.round(3), decimales=3, indice=True),
      "", al.tabla_md(dist_huecos.round(1), decimales=1, indice=True),
      "", f"### PM en las {VENTANA_PREVIA_H} h previas a cada hueco (>= {HUECO_MIN_H} h) frente al resto", "",
      al.tabla_md(q9.round(3), decimales=3, indice=True) if len(q9) else "Sin huecos.",
      "", "### Horas sobre el ECA", "", al.tabla_md(eca.round(2), indice=True),
      "", "### Fracción fina y PM2.5 > PM10", "", al.tabla_md(ff.round(3), decimales=3, indice=True),
      "", "### Ciclos (η² de la hora y del día de semana sobre log(1 + x))", "", al.tabla_md(ciclo.round(3), decimales=3, indice=True),
      "", "### Meteorología en las horas con PM (%)", "", al.tabla_md((100 * meteo).round(1), decimales=1, indice=True),
      "", "### Estaciones a menos de 15 km", "", al.tabla_md(parejas) if len(parejas) else "Ninguna.",
      "", "### Coordenadas y departamento", "",
      al.tabla_md(estaciones[[c for c in ["departamento", "provincia", "distrito", "lat", "lon", "juegos_de_ubicacion",
                                          "con_datos_validos"] if c in estaciones]], decimales=4, indice=True),
      "", "## Limpieza (prototipo de la cascada de Go)", "",
      "Orden de la cascada: " + " → ".join(al.NOMBRES_REGLAS) + ". Mismo orden y nombres que `datos/bloque.go`.", "",
      al.tabla_md(tabla_reglas, decimales=3),
      "", f"Leídas {al.miles(fl['leidas'])} = procesadas {al.miles(fl['procesadas'])} + descartadas {al.miles(fl['descartadas'])}. "
      f"Marcadas: {fl['marcadas_por_regla']}. Fuera de rango por parámetro: {fl['fuera_de_rango_por_parametro']}. "
      f"Filas anchas: {rep_limpieza['filas_anchas']}.",
      "", "## Supuestos del formato que hay que confirmar", "",
      f"- Config: {origen_cfg}" + (" + AJUSTES" if AJUSTES else "") + ".",
      f"- Columnas resueltas: `{info_lectura['columnas']}`.",
      f"- Parámetros sin canónico (descartados): {vistos['parametros_no_mapeados'] or 'ninguno'}.",
      f"- Periodos vistos: {vistos['periodos'] or 'sin columna de periodo'}.",
      f"- Unidades vistas: {vistos['unidades']}.",
      f"- Ejemplos de lo descartado: {rep_limpieza['_python']['ejemplos_por_regla']}.",
      f"- Delimitador {info_lectura['delimitador']!r}, codificación {info_lectura['codificacion']}, BOM {info_lectura['bom']}, "
      f"separador decimal {cfg.get('separador_decimal')!r}, fechas en [{cfg.get('fecha_desde')}, {cfg.get('fecha_hasta')}), "
      f"formatos {cfg.get('formatos_fecha')}.",
      f"- Primer MB: comillas impares en {primer_mb['lineas_con_comillas_impares']} líneas; columnas por fila {primer_mb['columnas_por_fila']}.",
      ]
(REPORTS / "eda_criba.md").write_text("\n".join(L) + "\n", encoding="utf-8")
print("Escrito", REPORTS / "eda_criba.md")
display(Markdown("\n".join(L[:14])))

## 14. Lo que hay que ajustar el lunes, con el CSV real

Todo va en `configs/limpieza.json` (el mismo que lee Go; probar antes con `AJUSTES`):

- **`columnas`**: si la celda 2 no encuentra alguna, poner el nombre real (el error sugiere uno).
  Si la hora viene en otra columna, `columnas.hora` (Go la une a la fecha con un espacio).
- **`formatos_fecha`**: layouts de Go que se prueban en orden (`02/01/2006 15:04` = día primero).
  Los ejemplos de `fecha` en la tabla de reglas dicen qué no se pudo leer. Go no lee `24:00`.
- **`parametros`**: completar con los nombres reales que quedaron sin canónico (tabla de vistos).
- **`periodos_horarios`**: confirmar qué valores son horarios.
- **`unidades`**: la de cada parámetro; una unidad que no figura descarta la fila.
- **`centinelas`, `rangos`, `fecha_desde`, `fecha_hasta`**: mirar los conteos de cada regla.
- **`separador_decimal`, `codificacion`, `delimitador`**: el primer MB los sugiere arriba.
- **Umbrales de la criba** (`U`): se reportan tal cual; si se mueven, el informe lo dice.